# Enterprise Data Quality Framework
### SQL + PySpark + Databricks

| Date | Author | Change |
|------------|---------------|--------------------------------------------------|
| 07-Jun-2026 | Sumit Saxena | Added 10 data quality checks for sql and pyspark |


Data quality issues can lead to:

- Incorrect business decisions
- Broken dashboards
- Failed ML models
- Regulatory compliance issues

This notebook demonstrates 10 essential data quality checks using both SQL and PySpark.


## Create Sample Data

This validation frameworks has used Banking Customer Onboarding & KYC Validation data as an example.

In [0]:
# Below data covers - Null Check, Duplicate Check and Accepted/Invalid Values Chaeck

customer_data = [
    (1001, "Rahul Sharma", "9876543210", "rahul@gmail.com", "PAN1234A", "ACTIVE"),
    (1002, "Priya Gupta", None, "priya@gmail.com", "PAN5678B", "ACTIVE"),      # Missing Mobile
    (1003, "Amit Singh", "9999999999", None, "PAN1111C", "ACTIVE"),            # Missing Email
    (1004, "Neha Jain", "8888888888", "neha@gmail.com", "PAN2222D", "UNKNOWN"), # Invalid Status
    (1005, "Rohit Verma", "7777777777", "rohit@gmail.com", "PAN3333E", "ACTIVE"),
    (1005, "Rohit Verma", "7777777777", "rohit@gmail.com", "PAN3333E", "ACTIVE") # Duplicate
]

customer_df = spark.createDataFrame(
    customer_data,
    ["customer_id","customer_name","mobile_no","email","pan_no","customer_status"]
)

customer_df.createOrReplaceTempView("customers")

display(customer_df)

In [0]:
# Covers Referential Integrity (customer_id=1006 doesn't exist) and Accepted Values (INVALID status) Checks

# from pyspark.sql import SparkSession
kyc_data = [
    (1001, "PAN", "VERIFIED", "2025-05-01"),
    (1002, "AADHAR", "VERIFIED", "2025-05-02"),
    (1003, "PAN", "PENDING", "2025-05-03"),
    (1006, "PAN", "VERIFIED", "2025-05-04"),  # Customer does not exist
    (1004, "VOTERID", "INVALID", "2025-05-05")
]

kyc_df = spark.createDataFrame(
    kyc_data,
    ["customer_id","document_type","verification_status","verification_date"]
)

kyc_df.createOrReplaceTempView("products")

display(kyc_df)

In [0]:
# Checks Covered Range Check, Accepted Values, Outlier Detection

account_data = [
    (5001,1001,"SAVINGS",50000),
    (5002,1002,"CURRENT",250000),
    (5003,1003,"SAVINGS",-1000),      # Negative Balance
    (5004,1004,"INVALID",50000),      # Invalid Account Type
    (5005,1005,"SAVINGS",100000000)   # Outlier Balance
]

account_df = spark.createDataFrame(
    account_data,
    ["account_no","customer_id","account_type","balance"]
)

account_df.createOrReplaceTempView("orders")
display(account_df)

In [0]:
# Checks Covered Referential Integrity, Accepted Values, Range Check, AML/Suspicious Transaction Detection

txn_data = [
    (9001,5001,10000,"CREDIT","2025-06-01"),
    (9002,5002,5000,"DEBIT","2025-06-02"),
    (9003,5003,-5000,"DEBIT","2025-06-03"),      # Negative Transaction
    (9004,9999,1000,"CREDIT","2025-06-04"),      # Invalid Account
    (9005,5001,50000000,"CREDIT","2025-06-05"),  # Suspicious Amount
    (9006,5002,2000,"INVALID","2025-06-06")      # Invalid Transaction Type
]

txn_df = spark.createDataFrame(
    txn_data,
    ["txn_id","account_no","txn_amount","txn_type","txn_date"]
)

txn_df.createOrReplaceTempView("events")

display(txn_df)

In [0]:
# Checks Temporal Consistency

kyc_renewal_data = [
    (1001,"2024-01-01","2027-01-01"),
    (1002,"2024-01-01","2023-01-01"), # Expiry before start
    (1003,"2024-01-01","2026-01-01")
]

kyc_renewal_df = spark.createDataFrame(
    kyc_renewal_data,
    ["customer_id","kyc_start_date","kyc_expiry_date"]
)

kyc_renewal_df.createOrReplaceTempView("events")

display(kyc_renewal_df)

In [0]:
subscriptions_data = [
    (1, "2025-01-01", "2025-12-31"),
    (2, "2025-05-01", "2025-04-01"),  # End Before Start
    (3, "2025-02-01", "2025-08-01")
]

subscriptions_df = spark.createDataFrame(
    subscriptions_data,
    ["subscription_id", "start_date", "end_date"]
)

subscriptions_df.createOrReplaceTempView("subscriptions")
display(subscriptions_df)

## 1: Null Check
Business Rule

Mandatory fields should never be null.

In [0]:
%sql
SELECT *
FROM products
WHERE product_id IS NULL
OR price IS NULL

In [0]:
products_df.filter(
    products_df.product_id.isNull()
).display()

## 2: Uniqueness Checks

In [0]:
%sql
SELECT product_id,
       COUNT(*)
FROM products
GROUP BY product_id
HAVING COUNT(*) > 1

In [0]:
from pyspark.sql.functions import count

products_df.groupBy("product_id") \
    .agg(count("*").alias("cnt")) \
    .filter("cnt > 1").display()

## 3: Referential Integrity


In [0]:
orphan_orders = (
    orders.alias("o")
    .join(
        customers.alias("c"),
        col("o.customer_id") == col("c.customer_id"),
        "left"
    )
    .filter(col("c.customer_id").isNull())
)

orphan_orders.show()

## 4. Accepted Values

In [0]:
valid_status = ["CREATED", "PAID", "SHIPPED", "CANCELLED"]

invalid_records = (
    orders.filter(~col("status").isin(valid_status))
)

invalid_records.show()

## 5. Functional Rules

In [0]:
invalid_products = (
    products.filter(
        (col("category") == "SHIRT") &
        (col("weight_kg") > 1)
    )
)

invalid_products.show()

## 6. Range Check

In [0]:
invalid_payments = (
    payments.filter(
        (col("amount") <= 0) |
        (col("amount") > 100000)
    )
)

invalid_payments.show()

## 7. Data Type Validation

In [0]:
from pyspark.sql.functions import regexp_extract

invalid_age = (
    users.filter(
        ~col("age").rlike("^[0-9]+$")
    )
)

invalid_age.show()

## 8. Freshness Check

In [0]:
from pyspark.sql.functions import max

latest_date = (
    events.agg(
        max("event_date").alias("max_dt")
    ).collect()[0]["max_dt"]
)

print(f"Latest Event Date: {latest_date}")

## 9. Temporal Consistency

In [0]:
invalid_dates = (
    subscriptions.filter(
        col("end_date") < col("start_date")
    )
)

invalid_dates.show()

## 10. Null Spike Detection

In [0]:
from pyspark.sql.functions import sum, when

total_count = users.count()

null_count = (
    users.select(
        sum(
            when(col("email").isNull(), 1)
            .otherwise(0)
        ).alias("null_cnt")
    )
    .collect()[0]["null_cnt"]
)

null_percentage = null_count / total_count

if null_percentage > 0.30:
    print("WARNING: Null spike detected")

## Display Data Quality Scorecard

In [0]:
dq_results = [
    ("Null Check", "PASS"),
    ("Uniqueness", "PASS"),
    ("Integrity", "FAIL"),
    ("Freshness", "PASS")
]

display(
    spark.createDataFrame(
        dq_results,
        ["Check","Status"]
    )
)

## Production Ready Function

In [0]:
class DataQualityFramework:

    def check_nulls(self, df, columns):
        pass

    def check_duplicates(self, df, keys):
        pass

    def check_referential_integrity(
            self,
            child_df,
            parent_df,
            child_key,
            parent_key):
        pass

    def check_accepted_values(
            self,
            df,
            column,
            valid_values):
        pass

    def check_range(
            self,
            df,
            column,
            min_val,
            max_val):
        pass

    def check_freshness(
            self,
            df,
            date_column):
        pass